# Customer-Feedback Intelligence Pipeline

## NLP Capstone Mini-Project

This notebook turns 40 inline Nimbus customer reviews into decision-ready feedback intelligence using three families on the same offline dataset:

1. **Classical NLP:** normalized text, TF-IDF, and logistic-regression sentiment.
2. **Semantic embeddings:** MiniLM vectors and cosine-similarity search.
3. **Small open-source LLM:** FLAN-T5 complaint summarization and zero/few-shot topic labeling.

The final outputs are a plain-words dashboard and a 200–300 word reflection comparing fit, quality, compute cost, and limitations.

**Professional deliverable name:** `customer_feedback_intelligence_pipeline.ipynb`

## Prerequisites: imports and fully inline dataset

No dataset file or URL is used. The sentence-transformer and FLAN-T5 models must already exist in the hosted lab's Hugging Face cache for fully offline execution. The notebook uses `local_files_only=True` and therefore will fail clearly rather than make a network request when model files are missing.

In [ ]:
import warnings, re, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RNG = 42
np.random.seed(RNG)

REVIEWS = [('The delivery arrived two days early and the packaging was flawless.', 'positive'), ('Battery life is incredible, easily lasts me a full working day.', 'positive'), ('Setup took five minutes and the app walked me through every step.', 'positive'), ('Customer support answered in under a minute and fixed my issue.', 'positive'), ('Build quality feels premium, the aluminium body is gorgeous.', 'positive'), ('Best purchase this year, the sound quality genuinely surprised me.', 'positive'), ('Screen is bright and crisp even in direct sunlight, love it.', 'positive'), ('Refund was processed instantly, no questions asked. Great service.', 'positive'), ('The subscription is worth every penny, new features every month.', 'positive'), ('Lightweight and comfortable, I forget I am even wearing it.', 'positive'), ('Pairing with my laptop was seamless and it just works.', 'positive'), ('Fantastic value for the price, would recommend to friends.', 'positive'), ('The firmware update fixed the lag and now it is buttery smooth.', 'positive'), ('Packaging was recyclable and the quick-start guide was clear.', 'positive'), ('Warranty claim was handled in a single email, very impressed.', 'positive'), ('Camera photos are sharp and colours look natural, thrilled.', 'positive'), ('Delivery was five days late and no one told me why.', 'negative'), ('The battery drains overnight even when the device is switched off.', 'negative'), ('App keeps crashing every time I open the settings menu.', 'negative'), ('Support put me on hold for forty minutes then dropped the call.', 'negative'), ('The plastic casing cracked within a week of normal use.', 'negative'), ('Overpriced for what it is, I expected far better at this cost.', 'negative'), ('Screen has a dead pixel right in the middle, very distracting.', 'negative'), ('They refused my refund and blamed me for the defect.', 'negative'), ('The subscription auto-renewed without any warning email.', 'negative'), ('It is heavy and uncomfortable after an hour of wearing it.', 'negative'), ('Bluetooth disconnects randomly every few minutes, infuriating.', 'negative'), ('Terrible value, it broke twice and shipping cost me extra.', 'negative'), ('The latest firmware update bricked my device completely.', 'negative'), ('Packaging was crushed and the manual was missing entirely.', 'negative'), ("Warranty was denied because the form was 'filled in wrong'.", 'negative'), ('Camera is grainy in low light and the autofocus hunts forever.', 'negative'), ('Shipping was quick but the box arrived dented on one corner.', 'negative'), ('Good sound but the app notifications are relentless and annoying.', 'negative'), ('Love the design though the battery could honestly be better.', 'positive'), ('Works well overall, only wish support replied a little faster.', 'positive'), ('Cheap feeling buttons let down an otherwise solid product.', 'negative'), ('The return shipping label never arrived so I am stuck with it.', 'negative'), ('Crystal clear calls and the mic cancels background noise well.', 'positive'), ('Constant software bugs make an expensive product feel like a beta.', 'negative')]

df = pd.DataFrame(REVIEWS, columns=["text", "sentiment"])
print(df.shape)
print(df["sentiment"].value_counts())
display(df.head(3))

assert df.shape == (40, 2)
assert set(df["sentiment"]) == {"positive", "negative"}

## Step 1: Normalize text for classical NLP

Raw text is retained for embedding and LLM tasks. The classical representation lowercases, removes punctuation, splits on whitespace, and drops a compact stop-word set.

In [ ]:
STOP = {
    "the","a","an","and","or","but","is","are","was","were","it","its","i",
    "me","my","of","to","in","for","on","at","this","that","with","as","so",
    "even","then","far","than","after","every","no","not","up","out","you",
}

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [token for token in text.split() if token and token not in STOP]

df["tokens"] = df["text"].apply(normalize)
df["clean"] = df["tokens"].apply(" ".join)

print("RAW :", df.loc[0, "text"])
print("TOKS:", df.loc[0, "tokens"])
print("avg tokens/review:", round(df["tokens"].str.len().mean(), 1))

## Step 2: TF-IDF + logistic-regression sentiment baseline

The split is stratified and fixed. With only 12 test reviews, accuracy is directional rather than production-grade. Coefficients offer transparent vocabulary-level interpretation.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

labels = (df["sentiment"] == "positive").astype(int)
Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(
    df["clean"], labels, df.index,
    test_size=0.30, stratify=labels, random_state=RNG
)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
Xtr_v = vec.fit_transform(Xtr)
Xte_v = vec.transform(Xte)
clf = LogisticRegression(C=1.0, max_iter=2000, random_state=RNG).fit(Xtr_v, ytr)
pred = clf.predict(Xte_v)
classical_accuracy = accuracy_score(yte, pred)

print(f"test accuracy = {classical_accuracy:.3f} (n_test={len(yte)})")
print("confusion matrix:
", confusion_matrix(yte, pred))
print(classification_report(yte, pred, target_names=["negative", "positive"], digits=3))

vocab = np.array(vec.get_feature_names_out())
coefs = clf.coef_[0]
top_pos = vocab[np.argsort(coefs)[-8:][::-1]]
top_neg = vocab[np.argsort(coefs)[:8]]
print("most POSITIVE features:", list(top_pos))
print("most NEGATIVE features:", list(top_neg))

### Inspect classical-model mistakes

Mixed sentiment is likely to be difficult because bag-of-words features cannot represent relationships between favorable and unfavorable clauses.

In [ ]:
held_out = df.loc[idx_te, ["text", "sentiment"]].copy()
held_out["predicted"] = np.where(pred == 1, "positive", "negative")
held_out["correct"] = held_out["sentiment"] == held_out["predicted"]
display(held_out)

mistakes = held_out[~held_out["correct"]]
print("Mistakes:", len(mistakes))
if len(mistakes):
    print("Example weak case:", mistakes.iloc[0]["text"])
else:
    print("No test error in this split; use the mixed battery/design review for nuance analysis.")

## Step 3: Semantic embeddings and similarity search

MiniLM creates normalized 384-dimensional vectors. Cosine similarity retrieves reviews that express related meaning even when vocabulary differs.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cpu",
    local_files_only=True,
)
emb = embedder.encode(
    df["text"].tolist(), normalize_embeddings=True, show_progress_bar=False
)
print("embedding matrix:", emb.shape)
assert emb.shape == (40, 384)

def search(query, k=3, exclude_exact=False):
    query_vector = embedder.encode([query], normalize_embeddings=True)
    similarities = cosine_similarity(query_vector, emb)[0]
    order = np.argsort(similarities)[::-1]
    results = []
    for index in order:
        if exclude_exact and df.loc[index, "text"] == query:
            continue
        results.append((round(float(similarities[index]), 3), df.loc[index, "text"]))
        if len(results) == k:
            break
    return results

for query in ["problems with the battery", "the shipping was slow"]:
    print("
Query:", query)
    for score, review in search(query):
        print(f"  {score:.3f}  {review}")

## Step 4: Summarize negative-review themes with FLAN-T5

The model is loaded from the local cache only. The generated summary is abstractive, so factual consistency must be checked against the source reviews.

In [ ]:
from transformers import pipeline

LLM_MODEL = "google/flan-t5-small"
generator = pipeline(
    "text2text-generation",
    model=LLM_MODEL,
    tokenizer=LLM_MODEL,
    device=-1,
    model_kwargs={"local_files_only": True},
)

negative_text = " ".join(df.loc[df["sentiment"] == "negative", "text"].tolist())
summary_prompt = (
    "Summarize the main problems customers complain about in these product reviews "
    "in one short paragraph. Mention only supported themes:
" + negative_text
)
complaint_summary = generator(
    summary_prompt, max_new_tokens=80, do_sample=False
)[0]["generated_text"]
print("SUMMARY OF COMPLAINTS:
", textwrap.fill(complaint_summary, 88))

## Step 5: Zero/few-shot topic labeling

A constrained prompt asks FLAN-T5 to choose exactly one topic. Two short examples provide few-shot guidance for the most confusing categories. Free text is snapped back to the allowed taxonomy.

In [ ]:
TOPICS = ["shipping", "battery", "app/software", "support", "build quality", "price"]

def label_topic(review):
    prompt = f"""Choose exactly one topic from: {', '.join(TOPICS)}.
Examples:
Review: The parcel came late and the box was damaged. Topic: shipping
Review: Customer service refused my return. Topic: support
Review: {review}
Topic:"""
    answer = generator(prompt, max_new_tokens=8, do_sample=False)[0]["generated_text"]
    normalized = answer.strip().lower()
    aliases = {
        "software": "app/software", "app": "app/software",
        "build": "build quality", "quality": "build quality",
        "delivery": "shipping", "refund": "support", "warranty": "support",
        "cost": "price", "value": "price",
    }
    for topic in TOPICS:
        if topic in normalized or topic.split("/")[0] in normalized:
            return topic
    for alias, topic in aliases.items():
        if alias in normalized:
            return topic
    return "other"

df["topic"] = df["text"].apply(label_topic)
display(df[["text", "sentiment", "topic"]].head(10))
print("Topic distribution:")
print(df["topic"].value_counts())

## Step 6: Plain-words stakeholder dashboard

Every figure traces to the original sentiment labels or generated topic classifications. The representative quote remains visible for audit.

In [ ]:
n = len(df)
pos = int((df["sentiment"] == "positive").sum())
neg = n - pos

print("=" * 64)
print("  NIMBUS - CUSTOMER FEEDBACK DASHBOARD (offline pipeline)")
print("=" * 64)
print(f"Reviews analysed      : {n}")
print(f"Sentiment split       : {pos} positive ({pos/n:.0%}) | {neg} negative ({neg/n:.0%})")

neg_by_topic = (
    df[df["sentiment"] == "negative"]
    .groupby("topic").size().sort_values(ascending=False)
)
print("
Top complaint areas (negative reviews by topic):")
for topic, count in neg_by_topic.items():
    print(f"  {topic:14s} {count:2d}  {'#' * int(count)}")

worst_topic = neg_by_topic.index[0]
example_review = df[
    (df["sentiment"] == "negative") & (df["topic"] == worst_topic)
].iloc[0]["text"]
print(f"
>> Loudest problem: '{worst_topic}'. Example: {example_review!r}")
print("
LLM complaint summary:")
print(textwrap.fill(complaint_summary, 88))

## Step 7: Same review, three analytical lenses

This comparison shows sentiment probability, semantic neighborhood, and instruction-based topic labeling for one mixed review.

In [ ]:
sample = "Love the design though the battery could honestly be better."
print("SAME REVIEW, THREE LENSES
" + "-" * 60)
print("Review:", sample, "
")

probability = clf.predict_proba(vec.transform([" ".join(normalize(sample))]))[0][1]
print(f"1) CLASSICAL (TF-IDF + LogReg) -> P(positive) = {probability:.2f}")

print("2) EMBEDDING (MiniLM search) -> closest other review:")
for score, review in search(sample, k=1, exclude_exact=True):
    print(f"   {score:.3f}  {review}")

sample_topic = label_topic(sample)
print(f"3) LLM (FLAN-T5 few-shot) -> topic = {sample_topic}")

## Step 8: Written analysis of classical NLP, embeddings, and LLMs

### A. Where TF-IDF was enough and where it fell short
TF-IDF with logistic regression was sufficient for a cheap, interpretable positive-versus-negative baseline. Its signed weights exposed useful surface cues such as praise, breakage, delays, and refund language. It was weaker for mixed reviews, paraphrases, and language where the relation between clauses matters.

### B. What embeddings added
MiniLM enabled semantic retrieval. A battery query could retrieve both “drains overnight” and “could be better,” even though the exact vocabulary differed. Embeddings supplied reusable meaning-based vectors, but the vectors did not independently produce a manager-ready summary or a controlled business taxonomy.

### C. What the small LLM added
FLAN-T5 converted many complaints into concise natural language and assigned topics without labeled topic-training data. This flexibility came with higher CPU latency and a risk of clipped, inconsistent, or unsupported output, so the generated result required constraints and source review.

### D. Mixed-sentiment nuance
For “Love the design though the battery could honestly be better,” no single family solved the entire problem. The classical model produced one overall probability, MiniLM found semantically related battery feedback, and FLAN-T5 selected the actionable battery topic. The combined pipeline handled the nuance better than any individual component.

### E. Cost and production choice
Compute need ranks TF-IDF/logistic regression lowest, MiniLM embeddings second, and FLAN-T5 generation highest. A production team should still choose TF-IDF when latency, throughput, interpretability, deterministic behavior, and tight CPU budgets matter more than open-ended language generation.

## Final Reflection (200–300 words)

For a team processing 10,000 reviews each week under a tight compute budget, I would keep the TF-IDF logistic-regression classifier as the default sentiment engine because it is fast, sparse, reproducible, and easy to retrain. I would compute MiniLM embeddings once per review and reuse those vectors for complaint search, clustering, and analyst investigations. I would not run FLAN-T5 independently on every review. Instead, I would use the small LLM on sampled or grouped complaint batches, such as the top negative clusters, because summarization and topic labeling are valuable but substantially slower and less deterministic than the classical model. This tiered design keeps high-volume scoring cheap while reserving generation for decisions that need natural-language synthesis.

The mixed review, “Love the design though the battery could honestly be better,” exposes a weakness. A binary classifier compresses two opinions into one label, while the topic model may focus only on battery and lose the positive design signal. I would first change the task to aspect-based sentiment, producing separate design and battery labels. A larger instruction-tuned model could improve nuance, but it would increase memory, latency, and validation cost; better labels and evaluation are more important than model size alone.

The dashboard’s loudest problem can justify a focused product decision, such as prioritizing a battery-drain investigation or changing shipping packaging and carrier controls. However, 40 reviews are too few for a confident portfolio-wide conclusion. The sample may overrepresent unusual failures, one product, or one time period. A product manager should validate the pattern against thousands of recent reviews, return reasons, support tickets, product versions, and category-level trends before reallocating budget.

## Submission checklist

- Run all cells top-to-bottom.
- Confirm the inline dataset contains 40 reviews.
- Confirm the TF-IDF metrics and feature weights are visible.
- Confirm the embedding matrix is `(40, 384)` and both searches return results.
- Confirm the complaint summary and topic distribution are visible.
- Confirm the plain-words dashboard prints.
- Keep the final reflection between 200 and 300 words.
- Save as `customer_feedback_intelligence_pipeline.ipynb`.